In [ ]:
import requests
import json
import time
import notebookutils
from notebookutils import mssparkutils
from pyspark.sql import SparkSession
from pyspark.sql.functions import max
from pyspark.sql.types import StructType, StructField, TimestampType, StringType
from datetime import datetime, timedelta, timezone
from dateutil import parser


StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 24, Finished, Available, Finished, False)

In [2]:
# Create Config Table in Lakehouse
# This table stores a single row with the last processed end time
config_table_name = "OAP_Load_Config"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {config_table_name} (
        LastEndTime TIMESTAMP,
        UpdatedAt TIMESTAMP
    )
    USING DELTA
""")

print(f"Verified configuration table: {config_table_name}")

StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 25, Finished, Available, Finished, False)

Verified configuration table: OAP_Load_Config


In [3]:
# Generate Time Windows for Processing (in-memory only, not stored in table)
# Use UTC for all time calculations
execution_start_time = datetime.now(timezone.utc)

# Read last processed end time from config table
df_config = spark.table(config_table_name)
max_row = df_config.agg(max("LastEndTime").alias("max_end")).collect()
last_end_time = max_row[0]["max_end"]

if last_end_time:
    # If table has data: back up 1 hour to capture late-arriving data
    last_end = last_end_time
    if last_end.tzinfo is None:
        last_end = last_end.replace(tzinfo=timezone.utc)
        
    current_start = last_end - timedelta(hours=1)
    print(f"Found existing data. Backing up 1h to overlap and catch late arrivals. Resuming from: {current_start}")
else:
    # Migrate Old Data (Table is empty): Start from 3 days ago
    print("No existing data found. Running migration logic for the last 3 days.")
    three_days_ago = execution_start_time - timedelta(days=3)
    current_start = three_days_ago.replace(hour=0, minute=0, second=0, microsecond=0)
    
    if current_start.tzinfo is None:
        current_start = current_start.replace(tzinfo=timezone.utc)
        
    print(f"Migration start date (aligned to midnight): {current_start}")

# Generate time windows (in-memory list)
# Constraint: A single window cannot cross midnight
windows_to_process = []

while current_start < execution_start_time:
    # Calculate next midnight
    next_day_date = current_start.date() + timedelta(days=1)
    next_midnight = datetime(next_day_date.year, next_day_date.month, next_day_date.day, 0, 0, 0, 0, tzinfo=timezone.utc)
    
    # Define limit: 1ms before midnight
    day_end_limit = next_midnight - timedelta(milliseconds=1)
    
    # The actual window end is the lesser of "End of Day" or "Current Time"
    window_end = min(day_end_limit, execution_start_time)
        
    # Safety break to prevent infinite loops
    if window_end <= current_start:
        break

    # Add window to in-memory list
    windows_to_process.append((current_start, window_end))
    
    # Advance start to the next millisecond
    current_start = window_end + timedelta(milliseconds=1)

if windows_to_process:
    print(f"Generated {len(windows_to_process)} time window(s) for processing.")
else:
    print("No new windows to process.")

StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 26, Finished, Available, Finished, False)

Found existing data. Backing up 1h to overlap and catch late arrivals. Resuming from: 2026-04-16 08:41:41.115000+00:00
Generated 1 time window(s) for processing.


In [ ]:
def get_tokens():
    """Acquire Fabric / Power BI and Kusto tokens via the notebook's run identity.

    Audiences:
      - 'pbi'   -> Fabric / Power BI REST APIs
      - 'kusto' -> Eventhouse / Kusto cluster REST API
    """
    global auth_token, kusto_token
    auth_token  = notebookutils.credentials.getToken('pbi')
    kusto_token = notebookutils.credentials.getToken('kusto')
    if auth_token and kusto_token:
        print('Successfully retrieved tokens.')
    else:
        raise Exception('Failed to retrieve essential tokens (PBI/Kusto).')

get_tokens()

headers = {
    'Authorization': f'Bearer {auth_token}',
    'Content-Type': 'application/json',
}


In [ ]:
# KQL Configuration
kusto_cluster_uri = 'https://trd-6uegjpfbf030eemxtw.z1.kusto.fabric.microsoft.com'
kusto_database    = 'MonitoringEventhouse'
target_table      = 'WorkspaceOutboundAccessProtection'
staging_table     = 'WorkspaceOutboundAccessProtection_Staging'

# kusto_token was acquired in the previous cell via get_tokens().

def run_kusto_command(query):
    """Execute a KQL Control Command via the Kusto REST API (no SDK needed)."""
    if not kusto_token:
        print('Cannot run command: No token available.')
        return

    mgmt_endpoint = f"{kusto_cluster_uri.rstrip('/')}/v1/rest/mgmt"
    headers = {
        'Authorization': f'Bearer {kusto_token}',
        'Content-Type': 'application/json'
    }
    body = {'db': kusto_database, 'csl': query}
    try:
        response = requests.post(mgmt_endpoint, headers=headers, json=body)
        response.raise_for_status()
        return True
    except Exception as e:
        print(f'Failed to execute KQL command via REST API: {e}')
        if 'response' in locals() and response.content:
            print(f'Details: {response.content}')
        return False


In [ ]:
# Define Processing Function
def process_activity_events_window(start_dt_obj, end_dt_obj):
    """
    Fetches data from PBI Activity API for the given window and loads it into KQL.
    Returns True if successful, False otherwise.
    """
    
    # 1. Format Dates for API (Single Quoted ISO String)
    # Ensure they are in UTC
    # Manually handle milliseconds to ensure wecapture the exact time (including .999) 
    # and strictly output 3 decimal places as preferred by many APIs.
    # %f produces 6 digits (microseconds), slicing [:-3] gives milliseconds.
    s_iso = start_dt_obj.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
    e_iso = end_dt_obj.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

    s_str = f"'{s_iso}'"
    e_str = f"'{e_iso}'"
    
    print(f"\n--- Processing Window: {s_str} to {e_str} ---")

    # 2. Get data from PBI REST API
    filters = [
        "Activity eq 'EnableWorkspaceOutboundAccessProtection'",
        "Activity eq 'DisableWorkspaceOutboundAccessProtection'"
    ]
    
    base_url = "https://api.powerbi.com/v1.0/myorg/admin/activityevents"
    all_local_events = []

    max_retries = 3
    retry_backoff_seconds = 5

    for f in filters:
        if not f or not f.strip():
            continue
        
        url = f"{base_url}?startDateTime={s_str}&endDateTime={e_str}&$filter={f}"
        
        while url:
            attempt = 0
            while True:
                attempt += 1
                try:
                    response = requests.get(url, headers=headers)
                    response.raise_for_status()
                    break  # success, exit retry loop
                except requests.exceptions.HTTPError as e:
                    status = getattr(e.response, 'status_code', None)
                    body = getattr(e.response, 'text', '')
                    print(f"HTTP {status} on filter '{f}' (attempt {attempt}/{max_retries}): {e} | body: {body[:500]}")
                    if attempt >= max_retries:
                        raise RuntimeError(
                            f"PBI Activity API failed after {max_retries} attempts "
                            f"(status={status}) for filter '{f}'. Aborting notebook."
                        ) from e
                    time.sleep(retry_backoff_seconds * attempt)
                except Exception as e:
                    print(f"Error on filter '{f}' (attempt {attempt}/{max_retries}): {e}")
                    if attempt >= max_retries:
                        raise RuntimeError(
                            f"PBI Activity API failed after {max_retries} attempts "
                            f"for filter '{f}'. Aborting notebook."
                        ) from e
                    time.sleep(retry_backoff_seconds * attempt)

            data = response.json()
            if 'activityEventEntities' in data:
                all_local_events.extend(data['activityEventEntities'])
            elif 'value' in data:
                all_local_events.extend(data['value'])

            url = data.get('continuationUri')

    print(f"Total events fetched: {len(all_local_events)}")

    if not all_local_events:
        print("No events found.")
        return True

    # 3. Load data to KQL Table
    try:
        # Create Spark DataFrame
        spark_df_local = spark.createDataFrame(all_local_events)
        
        #print(f"Writing {spark_df_local.count()} rows to Staging Table: {staging_table}")

        spark_df_local.write \
            .format("com.microsoft.kusto.spark.synapse.datasource") \
            .option("kustoCluster", kusto_cluster_uri) \
            .option("kustoDatabase", kusto_database) \
            .option("kustoTable", staging_table) \
            .option("accessToken", kusto_token) \
            .option("tableCreateOptions", "CreateIfNotExist") \
            .mode("Append") \
            .save()
            
        #print("Write to staging complete.")

        # Merge Logic
        merge_query = f"""
        .set-or-append {target_table} <| 
        {staging_table} 
        | join kind=leftanti {target_table} on Id
        """
        
        cleanup_query = f".clear table {staging_table} data"

        # Using the run_kusto_command helper defined in earlier cell
        if run_kusto_command(merge_query):
            print("Merge to target complete.")
            if run_kusto_command(cleanup_query):
                print("Staging cleanup complete.")
                return True
            else:
                print("Warning: Staging cleanup failed.")
                return False # Or True if you consider data loaded as success regardless of cleanup
        else:
            print("Merge failed.")
            return False
            
    except Exception as e:
        print(f"Error (Load/Merge): {e}")
        return False

StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 29, Finished, Available, Finished, False)

In [7]:
# Process Time Windows
if not windows_to_process:
    print("No time windows to process.")
else:
    total_items = len(windows_to_process)
    print(f"Starting processing of {total_items} window(s)...")
    
    last_successful_end = None
    
    for i, (window_start, window_end) in enumerate(windows_to_process):
        # Run processing logic
        success = process_activity_events_window(window_start, window_end)
        
        if success:
            # Track the last successful window end time (don't write to DB yet)
            last_successful_end = window_end
            
            # Wait 10s if not the last item
            if i < total_items - 1:
                print("Waiting 1s...")
                time.sleep(1)
        else:
            print(f"Processing failed for window {window_start} to {window_end}. Stopping loop.")
            break
    
    # Update config table once with the final successful end time
    if last_successful_end:
        try:
            new_watermark_str = last_successful_end.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
            spark.sql(f"""
                MERGE INTO {config_table_name} AS t
                USING (SELECT CAST('{new_watermark_str}' AS TIMESTAMP) AS LastEndTime) AS s
                ON (1 = 1)
                WHEN MATCHED THEN
                    UPDATE SET t.LastEndTime = s.LastEndTime, t.UpdatedAt = current_timestamp()
                WHEN NOT MATCHED THEN
                    INSERT (LastEndTime, UpdatedAt) VALUES (s.LastEndTime, current_timestamp())
            """)
            print(f"Watermark updated to: {new_watermark_str}")
        except Exception as e:
            print(f"Failed to update watermark in config table: {e}")
            
    print("Batch processing finished.")

StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 30, Finished, Available, Finished, False)

Starting processing of 1 window(s)...

--- Processing Window: '2026-04-16T08:41:41.115Z' to '2026-04-16T12:07:39.219Z' ---
Total events fetched: 1
Merge to target complete.
Staging cleanup complete.
Watermark updated to: 2026-04-16T12:07:39.219
Batch processing finished.


In [8]:
# Continuous Polling: Run every minute for 1 hour after historic load
# window_start = last_successful_end - 1h (or now - 1h on first iteration)
# window_end   = now
# Config table is updated ONCE after the final iteration

polling_duration_hours = 1          # Total time to run this loop
polling_interval_seconds = 60       # How often to poll (1 minute)

loop_start_time = datetime.now(timezone.utc)
loop_end_time = loop_start_time + timedelta(hours=polling_duration_hours)

print(f"Starting continuous polling from {loop_start_time.isoformat()} until {loop_end_time.isoformat()}")
print(f"Polling every {polling_interval_seconds}s. Window = [last_end - 1h, now]")

iteration = 0
last_successful_end_continuous = None

while True:
    now = datetime.now(timezone.utc)

    # Stop once 1 hour has elapsed
    if now >= loop_end_time:
        print(f"\nPolling duration of {polling_duration_hours}h reached. Stopping.")
        break

    iteration += 1

    # Use last successful end time as anchor; fall back to now on first iteration
    anchor = last_successful_end_continuous if last_successful_end_continuous is not None else now
    window_start = anchor - timedelta(hours=1)
    window_end = now

    print(f"\n[Iteration {iteration}] {now.strftime('%H:%M:%S UTC')} | window: [{window_start.strftime('%H:%M:%S')} - {window_end.strftime('%H:%M:%S')}]")
    success = process_activity_events_window(window_start, window_end)

    if success:
        last_successful_end_continuous = window_end
    else:
        print(f"Iteration {iteration} failed. Continuing to next interval.")

    # Check if loop time has expired before sleeping
    if datetime.now(timezone.utc) < loop_end_time:
        print(f"Sleeping {polling_interval_seconds}s until next iteration...")
        time.sleep(polling_interval_seconds)

# Update config table ONCE after the final iteration
if last_successful_end_continuous:
    try:
        new_watermark_str = last_successful_end_continuous.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
        spark.sql(f"""
            MERGE INTO {config_table_name} AS t
            USING (SELECT CAST('{new_watermark_str}' AS TIMESTAMP) AS LastEndTime) AS s
            ON (1 = 1)
            WHEN MATCHED THEN
                UPDATE SET t.LastEndTime = s.LastEndTime, t.UpdatedAt = current_timestamp()
            WHEN NOT MATCHED THEN
                INSERT (LastEndTime, UpdatedAt) VALUES (s.LastEndTime, current_timestamp())
        """)
        print(f"\nConfig table updated once with final watermark: {new_watermark_str}")
    except Exception as e:
        print(f"Failed to update config table: {e}")
else:
    print("\nNo successful iterations — config table not updated.")

print("Continuous polling finished.")


StatementMeta(, bf7749ca-d00f-4694-80e1-ae7c14de0134, 31, Cancelled, Cancelling, Cancelling, True)

Starting continuous polling from 2026-04-16T12:08:41.468148+00:00 until 2026-04-16T13:08:41.468148+00:00
Polling every 60s. Window = [last_end - 1h, now]

[Iteration 1] 12:08:41 UTC | window: [11:08:41 - 12:08:41]

--- Processing Window: '2026-04-16T11:08:41.468Z' to '2026-04-16T12:08:41.468Z' ---
Total events fetched: 1
Merge to target complete.
Staging cleanup complete.
Sleeping 60s until next iteration...

[Iteration 2] 12:10:37 UTC | window: [11:08:41 - 12:10:37]

--- Processing Window: '2026-04-16T11:08:41.468Z' to '2026-04-16T12:10:37.299Z' ---
Total events fetched: 1
Merge to target complete.
Staging cleanup complete.
Sleeping 60s until next iteration...

[Iteration 3] 12:12:15 UTC | window: [11:10:37 - 12:12:15]

--- Processing Window: '2026-04-16T11:10:37.299Z' to '2026-04-16T12:12:15.767Z' ---
Total events fetched: 1
Merge to target complete.
Staging cleanup complete.
Sleeping 60s until next iteration...

[Iteration 4] 12:13:56 UTC | window: [11:12:15 - 12:13:56]

--- Process